# Reasoning Benchmark Datasets: ProofWriter OWA, ContractNLI, SARA, OpenBookQA

This notebook demonstrates the dataset collection and standardization pipeline for evaluating a neuro-symbolic FOL translation pipeline. Four benchmark datasets are unified into a common schema (`exp_sel_data_out`) for comparison:

1. **ProofWriter OWA** – Multi-hop logical reasoning with three-valued labels (True/False/Unknown)
2. **ContractNLI** – Legal NLI over NDA contract clauses (Entailment/Contradiction/NotMentioned)
3. **SARA** – US federal tax statutory reasoning with gold Prolog predicate annotations
4. **OpenBookQA** – Science multi-hop QA combining a core science fact with reading comprehension

The demo loads a curated mini subset (12 examples, 3 per dataset) from GitHub.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab
_pip('loguru==0.7.2')

# Core packages pre-installed on Colab; install locally to match Colab versions
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
import json
import re
import random
from pathlib import Path
import urllib.request
import os

import matplotlib.pyplot as plt
import pandas as pd

random.seed(42)

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-45095e-provenance-stratified-neuro-symbolic-rea/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("Loaded metadata:", data["metadata"]["description"])
print("Total examples (full dataset):", data["metadata"]["total_examples"])
print("Datasets in mini:", [d["dataset"] for d in data["datasets"]])

## Configuration

Tunable parameters for the dataset builders. Set to minimum values for the demo; original full-run values are shown in comments.

In [ ]:
# Maximum examples to sample per dataset (demo: 3 per dataset = 12 total)
# Original full-run: PROOFWRITER_MAX = 5000, SNLI_MAX = 2000
PROOFWRITER_MAX = 3   # original: 5000
SNLI_MAX = 3          # original: 2000
RANDOM_SEED = 42

## ProofWriter OWA

Multi-hop natural language logical reasoning with three-valued labels (True/False/Unknown) under the Open World Assumption. Each example has a natural language theory (facts + rules) and a yes/no/unknown question. The script balances labels by sampling `max_examples // 3` per label.

In [ ]:
# Extract ProofWriter examples from loaded mini data
proofwriter_ds = next(d for d in data["datasets"] if d["dataset"] == "proofwriter_owa")
raw = proofwriter_ds["examples"]

# Simulate the balancing logic from build_proofwriter()
by_label = {}
for r in raw:
    lbl = r["output"]
    if lbl in ("True", "False", "Unknown"):
        by_label.setdefault(lbl, []).append(r)

per_label = PROOFWRITER_MAX // 3
sampled = []
for lbl, rows in by_label.items():
    sampled.extend(random.sample(rows, min(per_label or 1, len(rows))))
random.shuffle(sampled)
proofwriter_examples = sampled[:PROOFWRITER_MAX] if PROOFWRITER_MAX > 0 else raw

print(f"ProofWriter examples: {len(proofwriter_examples)}")
for ex in proofwriter_examples[:2]:
    print(f"  [{ex['output']}] hop_count={ex['metadata_hop_count']} | {ex['input'][:80]}...")

## ContractNLI

Document-level NLI over NDA contract clauses. Labels: Entailment / Contradiction / NotMentioned. Each example pairs a contract clause (premise) with a hypothesis about confidentiality obligations.

In [ ]:
contractnli_ds = next(d for d in data["datasets"] if d["dataset"] == "contractnli")
contractnli_examples = contractnli_ds["examples"]

label_map = {0: "NotMentioned", 1: "Entailment", 2: "Contradiction"}

print(f"ContractNLI examples: {len(contractnli_examples)}")
for ex in contractnli_examples[:2]:
    print(f"  [{ex['output']}] split={ex['metadata_split']} | {ex['input'][:80]}...")

## SARA

US federal tax law statutory reasoning. Each example is a natural language case description paired with a yes/no tax obligation question. Gold Prolog predicate annotations are included in `metadata_gold_predicates` for Phase 0 calibration of the neuro-symbolic pipeline.

In [ ]:
sara_ds = next(d for d in data["datasets"] if d["dataset"] == "sara")
sara_examples = sara_ds["examples"]

print(f"SARA examples: {len(sara_examples)}")
for ex in sara_examples[:2]:
    predicates = json.loads(ex.get("metadata_gold_predicates", "[]"))
    print(f"  [{ex['output']}] section={ex['metadata_statute_section']} | predicates: {predicates[:2]}")
    print(f"  Input: {ex['input'][:80]}...")

## OpenBookQA

Science multi-hop QA requiring combination of a core science fact with reading comprehension. Each example has a core fact, a question, and 4 answer choices. Hop count is fixed at 2 (core fact + comprehension step).

In [ ]:
openbookqa_ds = next(d for d in data["datasets"] if d["dataset"] == "openbookqa")
openbookqa_examples = openbookqa_ds["examples"]

print(f"OpenBookQA examples: {len(openbookqa_examples)}")
for ex in openbookqa_examples[:2]:
    print(f"  [{ex['output']}] core_fact: {ex.get('metadata_core_fact', '')[:60]}")
    print(f"  Input: {ex['input'][:80]}...")

## Assemble Unified Dataset

All four datasets are merged into a single list following the `exp_sel_data_out` schema. Each example has `input`, `output`, and `metadata_*` fields.

In [ ]:
# Replicate the main() assembly logic from data.py
all_examples_by_dataset = [
    {"dataset": "proofwriter_owa", "examples": proofwriter_examples},
    {"dataset": "contractnli",     "examples": contractnli_examples},
    {"dataset": "sara",            "examples": sara_examples},
    {"dataset": "openbookqa",      "examples": openbookqa_examples},
]

total = sum(len(d["examples"]) for d in all_examples_by_dataset)

result = {
    "metadata": {
        "description": "Neuro-symbolic reasoning benchmark: ProofWriter OWA, ContractNLI, SARA, OpenBookQA",
        "total_examples": total,
        "hypothesis": "FOL translation pipeline for multi-hop reasoning over textual documents",
    },
    "datasets": all_examples_by_dataset,
}

print(f"Total examples in demo: {total}")
for ds in all_examples_by_dataset:
    print(f"  {ds['dataset']}: {len(ds['examples'])} examples")

## Visualization & Summary

Summary table and chart of the dataset collection.

In [ ]:
# Build summary table
rows = []
for ds in all_examples_by_dataset:
    exs = ds["examples"]
    labels = [e["output"] for e in exs]
    unique_labels = sorted(set(labels))
    domains = sorted(set(e["metadata_domain"] for e in exs))
    task_types = sorted(set(e["metadata_task_type"] for e in exs))
    hop_counts = [e["metadata_hop_count"] for e in exs]
    rows.append({
        "Dataset": ds["dataset"],
        "# Examples (demo)": len(exs),
        "Domain": ", ".join(domains),
        "Task Type": ", ".join(task_types),
        "Labels": ", ".join(unique_labels),
        "Avg Hop Count": round(sum(hop_counts) / len(hop_counts), 2) if hop_counts else 0,
    })

df = pd.DataFrame(rows)
print("=== Dataset Summary ===")
print(df.to_string(index=False))

# Bar chart: examples per dataset
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df["Dataset"], df["# Examples (demo)"], color=["steelblue", "darkorange", "green", "crimson"])
ax.set_title("Examples per Dataset (Demo Subset)")
ax.set_ylabel("Count")
ax.set_xlabel("Dataset")
plt.tight_layout()
plt.savefig("dataset_summary.png", dpi=100)
plt.show()
print("\nFull dataset sizes (from metadata):", data["metadata"]["total_examples"], "total examples")